# F1 pit-stop strategy predictor

For every lap of a race the model answers two questions a strategist asks:

1. **Should this driver pit on this lap?** (binary, ~3 % of laps are pit laps)
2. **If they pit, which compound goes on?** (Soft / Medium / Hard / Intermediate / Wet)


## 0. Setup

In [ ]:
%pip install -q fastf1 lightgbm

In [ ]:
import os
import glob
import time
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from tqdm.auto import tqdm

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.width', 140)

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/F1_Project'
except ImportError:
    BASE_DIR = os.environ.get('F1_PROJECT_DIR', './F1_Project')

CACHE_DIR = os.path.join(BASE_DIR, 'f1_cache')
SAVE_DIR = os.path.join(BASE_DIR, 'processed_races')
OUT_DIR = os.path.join(BASE_DIR, 'results')
for folder in [CACHE_DIR, SAVE_DIR, OUT_DIR]:
    os.makedirs(folder, exist_ok=True)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__} on {device}'
      + (f' ({torch.cuda.get_device_name(0)})' if device.type == 'cuda' else ''))

# plot style used by every figure
COLORS = {'net': '#2a78d6', 'gbm': '#eb6834', 'original': '#1baf7a', 'rule': '#8c8b85'}
TYRE_COLORS = {'SOFT': '#e34948', 'MEDIUM': '#eda100', 'HARD': '#8c8b85',
               'INTERMEDIATE': '#1baf7a', 'WET': '#2a78d6'}
INK, MUTED, GRID = '#1f1f1d', '#6b6a66', '#e6e5e0'
plt.rcParams.update({
    'figure.dpi': 110, 'savefig.dpi': 160, 'savefig.bbox': 'tight',
    'font.size': 10, 'axes.titlesize': 12, 'axes.titleweight': 'semibold', 'axes.titlelocation': 'left',
    'axes.labelsize': 10, 'axes.labelcolor': INK, 'axes.edgecolor': MUTED, 'axes.linewidth': 0.8,
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True,
    'grid.color': GRID, 'grid.linewidth': 0.8, 'xtick.color': MUTED, 'ytick.color': MUTED,
    'legend.frameon': False, 'lines.linewidth': 2,
})


def save(fig, name):
    fig.savefig(os.path.join(OUT_DIR, f'{name}.png'))

## 1. Data download
One CSV per race (FastF1 laps + weather). Already downloaded files are skipped; the API allows ~500 calls per hour.

In [ ]:
def prepare_laps(laps):
    laps = laps.sort_values(['Driver', 'LapNumber'])
    laps['LapTimeSec'] = laps['LapTime'].dt.total_seconds()
    laps['Rainfall'] = laps['Rainfall'].astype(int)
    laps['PitThisLap'] = laps['PitInTime'].notna().astype(int)
    for col in ['LapTimeSec', 'Position', 'TrackStatus', 'AirTemp', 'TrackTemp', 'Rainfall']:
        laps[f'Prev_{col}'] = laps.groupby('Driver')[col].shift(1)
    return laps.dropna(subset=['Prev_LapTimeSec'])


def download_races(first_year, last_year):
    import fastf1
    fastf1.Cache.enable_cache(CACHE_DIR)
    schedule = []
    for year in range(first_year, last_year + 1):
        events = fastf1.get_event_schedule(year)
        events = events[events['EventFormat'] != 'testing']
        schedule += [(year, int(r.RoundNumber), r.EventName) for r in events.itertuples()]

    for year, rnd, name in tqdm(schedule, desc='Downloading'):
        path = os.path.join(SAVE_DIR, f'{year}_{rnd}.csv')
        if os.path.exists(path):
            continue
        try:
            session = fastf1.get_session(year, rnd, 'R')
            session.load(laps=True, telemetry=False, weather=True, messages=False)
            laps = session.laps.copy()
            if len(laps) == 0:
                continue
            laps = pd.merge_asof(laps.sort_values('Time'), session.weather_data.sort_values('Time'),
                                 on='Time', direction='backward')
            laps = prepare_laps(laps)
            laps['Year'], laps['Event'] = year, name
            laps.to_csv(path, index=False)
            time.sleep(1)
        except Exception as err:
            if 'RateLimitExceeded' in str(err) or '500 calls/h' in str(err):
                print(f'API rate limit reached at {year} {name}, run this cell again in an hour.')
                break
            print(f'Skipping {year} {name}: {err}')


n_files = len(glob.glob(os.path.join(SAVE_DIR, '*.csv')))
if n_files < 170:
    download_races(2018, 2025)
else:
    print(f'{n_files} races already downloaded, skipping.')

## 2. Loading and cleaning
- **2018 compounds:** that season used absolute names (hypersoft … superhard). Each race had three, so they are ranked by hardness and relabelled Soft/Medium/Hard like 2019+.
- **Lap grid:** some laps are missing in the raw files, so every driver is reindexed to consecutive laps. `shift(1)` then really means *previous lap*.
- **Retirements:** a pit entry on a driver's last lap is a retirement, not a strategy call, and is removed from the targets.

In [ ]:
DRY = ['SOFT', 'MEDIUM', 'HARD']
COMPOUNDS = ['SOFT', 'MEDIUM', 'HARD', 'INTERMEDIATE', 'WET']

# 2018 used absolute names (hypersoft ... superhard). Pirelli brought three per race,
# so we rank them by hardness and relabel them SOFT / MEDIUM / HARD like 2019+.
HARDNESS_2018 = ['HYPERSOFT', 'ULTRASOFT', 'SUPERSOFT', 'SOFT', 'MEDIUM', 'HARD', 'SUPERHARD']

# same track under a different race name, share the circuit history
CIRCUIT_ALIASES = {'São Paulo Grand Prix': 'Brazilian Grand Prix', 'Mexico City Grand Prix': 'Mexican Grand Prix',
                   '70th Anniversary Grand Prix': 'British Grand Prix', 'Styrian Grand Prix': 'Austrian Grand Prix'}

KEEP_COLS = ['Year', 'Event', 'Driver', 'Team', 'LapNumber', 'Stint', 'TyreLife', 'FreshTyre',
             'Compound', 'LapTimeSec', 'Time', 'PitInTime', 'PitOutTime', 'TrackStatus',
             'Position', 'Rainfall', 'TrackTemp', 'AirTemp', 'Humidity', 'PitThisLap']


def normalize_compounds(race):
    comp = race['Compound'].astype(object).str.upper()
    if race['Year'].iloc[0] <= 2018:
        used = [c for c in HARDNESS_2018 if c in set(comp.dropna())]
        if len(used) == 3:
            mapping = dict(zip(used, DRY))
        else:
            mapping = {c: ('SOFT' if i < 4 else 'MEDIUM' if i == 4 else 'HARD')
                       for i, c in enumerate(HARDNESS_2018)}
        comp = comp.replace(mapping)
    comp = comp.where(comp.isin(COMPOUNDS))
    # an unknown compound inside a stint is the same set of tyres
    race['Compound'] = comp.groupby([race['Driver'], race['Stint']]).transform(
        lambda s: s.ffill().bfill())
    return race


def to_lap_grid(race):
    """Reindex every driver to consecutive laps, so shift(1) really means 'previous lap'."""
    pieces = []
    for driver, laps in race.groupby('Driver'):
        laps = laps.drop_duplicates('LapNumber').set_index('LapNumber')
        full = laps.reindex(np.arange(laps.index.min(), laps.index.max() + 1))
        full['observed'] = full['Driver'].notna()
        full['Driver'] = driver
        pieces.append(full.rename_axis('LapNumber').reset_index())
    grid = pd.concat(pieces, ignore_index=True)
    for col in ['Year', 'Event']:
        grid[col] = race[col].iloc[0]
    grid['Team'] = grid.groupby('Driver')['Team'].transform(lambda s: s.ffill().bfill())
    grid['Stint'] = grid.groupby('Driver')['Stint'].transform(lambda s: s.ffill().bfill())
    return grid


def load_race(path):
    race = pd.read_csv(path, low_memory=False)
    race = race[[c for c in KEEP_COLS if c in race.columns]].dropna(subset=['Driver', 'LapNumber']).copy()
    race['LapNumber'] = race['LapNumber'].astype(int)
    race['Stint'] = race.groupby('Driver')['Stint'].transform(lambda s: s.ffill().bfill())
    race = normalize_compounds(race)
    race['time_s'] = pd.to_timedelta(race['Time']).dt.total_seconds()
    race['status'] = race['TrackStatus'].apply(lambda x: '' if pd.isna(x) else str(int(x)))
    race = to_lap_grid(race.drop(columns=['Time', 'TrackStatus']))
    race['PitThisLap'] = race['PitThisLap'].fillna(0).astype(int)
    race['total_laps'] = race['LapNumber'].max()
    return race


def load_all_races(folder):
    paths = sorted(glob.glob(os.path.join(folder, '*.csv')))
    races, failed = [], []
    for path in tqdm(paths, desc='Loading races'):
        try:
            races.append(load_race(path))
        except Exception as err:
            failed.append((os.path.basename(path), repr(err)))
    if failed:
        print(f'Skipped {len(failed)} files:', failed[:5])
    laps = pd.concat(races, ignore_index=True)
    laps['race_id'] = laps['Year'].astype(str) + ' ' + laps['Event']
    laps['circuit'] = laps['Event'].replace(CIRCUIT_ALIASES)
    return laps


laps = load_all_races(SAVE_DIR)
print(f'{laps.race_id.nunique()} races, {laps.observed.sum():,} observed laps, '
      f'{laps.PitThisLap.sum():,} pit laps')

## 3. Features
All values are known when the lap starts (or, for SC/VSC, shown live by race control):

| Group | Examples |
|---|---|
| Tyres & stint | tyre age, compound, compounds used so far, *needs a second dry compound* |
| Pace & degradation | fuel-corrected pace loss vs. stint best, degradation slope, pace vs. field |
| Rivals & field | gaps ahead/behind, did the car ahead/behind just pit, share of field that already stopped more, compounds the field is fitting |
| Safety car | SC/VSC last lap, SC/VSC now, freshly deployed neutralisation |
| Weather | rain, slick-vs-intermediate lap-time crossover |
| Circuit history | usual stint length, stops and pit-lane loss at this circuit **from earlier seasons only** |

In [ ]:
FUEL_EFFECT = 0.06   # s/lap: a car gets roughly this much faster per lap as fuel burns off
GAP_CAP = 40.0       # gaps beyond ~a pit-stop loss all mean "clear air"


def rolling_slope(x, y, window):
    """Least-squares slope of y on x over a trailing window, ignoring NaN."""
    mask = x.notna() & y.notna()
    x, y = x.where(mask), y.where(mask)
    roll = lambda s: s.rolling(window, min_periods=3)
    n = roll(x).count()
    sx, sy = roll(x).sum(), roll(y).sum()
    sxy, sxx = roll(x * y).sum(), roll(x * x).sum()
    denom = n * sxx - sx ** 2
    return ((n * sxy - sx * sy) / denom).where(denom > 0)


def add_lap_info(df):
    """Things we learn at the END of each lap. They are only used shifted by one lap."""
    status = df['status'].fillna('')
    df['sc'] = status.str.contains('4').astype(int)
    df['vsc'] = status.str.contains('6|7').astype(int)

    clean = (df['LapTimeSec'].notna() & df['PitInTime'].isna() & df['PitOutTime'].isna()
             & status.isin(['1', '2', '12', '21']) & (df['LapNumber'] > 1))
    field_median = df['LapTimeSec'].where(clean).groupby([df['race_id'], df['LapNumber']]).transform('median')
    clean &= df['LapTimeSec'] < 1.07 * field_median          # drop traffic, spins, mistakes
    df['clean_time'] = df['LapTimeSec'].where(clean)
    df['fuel_corr_time'] = df['clean_time'] + FUEL_EFFECT * df['LapNumber']
    df['pace_vs_field'] = df['clean_time'] - field_median

    # time lost by a green-flag stop: in-lap + out-lap compared with a normal lap
    df['field_median'] = field_median
    by_driver = df.groupby(['race_id', 'Driver'])
    green_stop = (df['PitThisLap'] == 1) & (df['status'] == '1') & (by_driver['status'].shift(-1) == '1')
    in_lap = df['LapTimeSec'] - df['field_median']
    out_lap = by_driver['LapTimeSec'].shift(-1) - by_driver['field_median'].shift(-1)
    df['pit_loss'] = (in_lap + out_lap).where(green_stop)

    # crossing order on each lap gives the cars directly ahead / behind
    timed = df[df['time_s'].notna()].sort_values(['race_id', 'LapNumber', 'time_s'])
    crossing = timed.groupby(['race_id', 'LapNumber'])
    df['gap_ahead'] = (timed['time_s'] - crossing['time_s'].shift(1)).clip(0, GAP_CAP).fillna(GAP_CAP)
    df['gap_behind'] = (crossing['time_s'].shift(-1) - timed['time_s']).clip(0, GAP_CAP).fillna(GAP_CAP)
    df['car_ahead'] = crossing['Driver'].shift(1)
    df['car_behind'] = crossing['Driver'].shift(-1)

    # did the car that was ahead/behind at the end of the previous lap pit on this lap?
    pitted = df.set_index(['race_id', 'LapNumber', 'Driver'])['PitThisLap']
    for side in ['ahead', 'behind']:
        rival = df.groupby(['race_id', 'Driver'])[f'car_{side}'].shift(1)
        keys = pd.MultiIndex.from_arrays([df['race_id'], df['LapNumber'], rival])
        df[f'{side}_pitted'] = pitted.reindex(keys).fillna(0).values

    # mixed conditions: how the cars on wet tyres compare with the cars on slicks
    on_wets = df['Compound'].isin(['INTERMEDIATE', 'WET'])
    racing_lap = df['LapTimeSec'].where(df['PitInTime'].isna() & df['PitOutTime'].isna())
    keys = [df['race_id'], df['LapNumber']]
    wets_time = racing_lap.where(on_wets).groupby(keys).transform('median')
    slicks_time = racing_lap.where(~on_wets & df['Compound'].notna()).groupby(keys).transform('median')
    df['wet_crossover'] = (slicks_time - wets_time).clip(-20, 20)

    # which tyres the rest of the field has fitted at their stops so far in this race
    fitted = by_driver['Compound'].shift(-1).where(df['PitThisLap'] == 1)
    all_laps = pd.MultiIndex.from_frame(df[['race_id', 'LapNumber']].drop_duplicates().sort_values(['race_id', 'LapNumber']))
    per_lap = (pd.crosstab([df['race_id'], df['LapNumber']], fitted)
               .reindex(index=all_laps, columns=COMPOUNDS, fill_value=0))
    so_far = per_lap.groupby(level=0).cumsum()
    total = so_far.sum(axis=1)
    for comp in ['SOFT', 'MEDIUM', 'HARD', 'INTERMEDIATE']:
        share = (so_far[comp] / total).where(total > 0).rename(f'fitted_{comp.lower()}')
        df = df.join(share, on=['race_id', 'LapNumber'])

    lap_groups = df.groupby(['race_id', 'LapNumber'])
    df['field_wet_share'] = on_wets.astype(float).where(df['Compound'].notna()).groupby(keys).transform('mean')
    df['field_pits'] = lap_groups['PitThisLap'].transform('sum')
    df['field_size'] = lap_groups['observed'].transform('sum')
    df['team_pits'] = df.groupby(['race_id', 'LapNumber', 'Team'])['PitThisLap'].transform('sum') - df['PitThisLap']
    return df


def add_features(df):
    df = add_lap_info(df)
    by_driver = df.groupby(['race_id', 'Driver'])
    prev = lambda col: by_driver[col].shift(1)

    # race and tyre state, known before the lap starts
    df['lap'] = df['LapNumber']
    df['race_progress'] = df['LapNumber'] / df['total_laps']
    df['laps_left'] = df['total_laps'] - df['LapNumber']
    df['tyre_age'] = df['TyreLife'].fillna(by_driver['TyreLife'].shift(1) + 1)
    df['stops_done'] = (df['Stint'] - 1).clip(lower=0)
    df['fresh_tyre'] = df['FreshTyre'].map({True: 1, False: 0, 'True': 1, 'False': 0})
    for comp in COMPOUNDS:
        df[f'on_{comp.lower()}'] = (df['Compound'] == comp).astype(int)

    # which compounds have been used so far (dry races need two different dry ones)
    for comp in COMPOUNDS:
        df[f'used_{comp.lower()}'] = df.groupby(['race_id', 'Driver'])[f'on_{comp.lower()}'].cummax()
    df['dry_used'] = df[['used_soft', 'used_medium', 'used_hard']].sum(axis=1)
    wet_race_so_far = (df['used_intermediate'] + df['used_wet']) > 0
    df['needs_second_dry'] = ((df['dry_used'] < 2) & ~wet_race_so_far).astype(int)

    # pace and degradation in the current stint, using laps that are already finished
    by_stint = df.groupby(['race_id', 'Driver', 'Stint'])
    df['stint_best'] = by_stint['fuel_corr_time'].transform(lambda s: s.cummin().ffill().shift(1))
    df['last_clean'] = by_stint['fuel_corr_time'].transform(lambda s: s.ffill().shift(1))
    df['pace_delta'] = df['last_clean'] - df['stint_best']
    slope = by_stint.apply(lambda g: rolling_slope(g['LapNumber'].astype(float), g['fuel_corr_time'], 6),
                           include_groups=False)
    df['deg_rate'] = slope.reset_index(level=[0, 1, 2], drop=True).sort_index()
    df['deg_rate'] = df.groupby(['race_id', 'Driver', 'Stint'])['deg_rate'].shift(1).clip(-1, 1)
    df['pace_vs_field'] = prev('pace_vs_field')

    # race situation at the end of the previous lap
    df['position'] = prev('Position')
    df['gap_ahead'] = prev('gap_ahead')
    df['gap_behind'] = prev('gap_behind')
    df['field_pit_share'] = prev('field_pits') / prev('field_size')
    df['field_pits_3'] = by_driver['field_pits'].transform(lambda s: s.shift(1).rolling(3, min_periods=1).sum())
    df['teammate_pitted'] = prev('team_pits').clip(upper=1)
    for comp in ['soft', 'medium', 'hard', 'intermediate']:
        df[f'field_fitted_{comp}'] = prev(f'fitted_{comp}').fillna(0)
    df['ahead_pitted'] = prev('ahead_pitted')
    df['behind_pitted'] = prev('behind_pitted')
    # share of the field that has already made more stops than this driver
    df['field_ahead_on_stops'] = df.groupby(['race_id', 'LapNumber'])['stops_done'].transform(
        lambda s: (s.values[None, :] > s.values[:, None]).mean(axis=1))

    # track status: last lap, plus what race control shows during this lap
    df['sc_last_lap'] = prev('sc')
    df['vsc_last_lap'] = prev('vsc')
    df['sc_now'] = df['sc']
    df['vsc_now'] = df['vsc']
    df['new_neutralisation'] = (((df['sc_now'] + df['vsc_now']) > 0)
                                & ((df['sc_last_lap'] + df['vsc_last_lap']) == 0)).astype(int)

    # weather (station data, previous lap)
    df['rain'] = prev('Rainfall')
    df['track_temp'] = prev('TrackTemp')
    df['humidity'] = prev('Humidity')
    on_wets = df['on_intermediate'] + df['on_wet']
    df['tyre_weather_mismatch'] = (((df['rain'] == 1) & (on_wets == 0))
                                   | ((df['rain'] == 0) & (on_wets == 1))).astype(int)
    # positive when the other tyre type is faster: wets for a car on slicks and vice versa
    df['crossover_gain'] = (prev('wet_crossover') * np.where(on_wets == 1, -1, 1)).fillna(0)
    df['odd_one_out'] = (prev('field_wet_share') - on_wets).abs()
    return df


def add_targets(df):
    """pit: did the driver pit on this lap; next_compound: compound of the next stint."""
    stints = (df[df['observed']].groupby(['race_id', 'Driver', 'Stint'])['Compound']
              .agg(lambda s: s.dropna().iloc[0] if s.notna().any() else np.nan).rename('stint_compound')
              .reset_index())
    stints['next_compound'] = stints.groupby(['race_id', 'Driver'])['stint_compound'].shift(-1)
    df = df.merge(stints[['race_id', 'Driver', 'Stint', 'next_compound']],
                  on=['race_id', 'Driver', 'Stint'], how='left')

    # not every pit entry is a tyre stop: retiring in the pits (last lap), drive-through penalties
    # and SC laps routed through the pit lane (same tyres, tyre age keeps counting)
    by_driver = df.groupby(['race_id', 'Driver'])
    last_lap = by_driver['LapNumber'].transform('max')
    same_tyres = ((by_driver['TyreLife'].shift(-1) == df['TyreLife'] + 1)
                  & (by_driver['Compound'].shift(-1) == df['Compound']))
    df['retired_in_pits'] = (df['PitThisLap'] == 1) & (df['LapNumber'] == last_lap)
    df['no_tyre_change'] = (df['PitThisLap'] == 1) & same_tyres & ~df['retired_in_pits']
    df['pit'] = df['PitThisLap']
    df['next_compound_id'] = df['next_compound'].map({c: i for i, c in enumerate(COMPOUNDS)})
    return df


laps = add_targets(add_features(laps))
data = laps[laps['observed'] & ~laps['retired_in_pits'] & ~laps['no_tyre_change']
            & (laps['LapNumber'] > 1)].reset_index(drop=True)
print(f'{len(data):,} laps for modelling, pit rate {data.pit.mean():.2%} | removed pit entries: '
      f'{laps.retired_in_pits.sum()} retirements, {laps.no_tyre_change.sum()} without a tyre change')

## 4. Season split and circuit priors

In [ ]:
VAL_YEAR, TEST_YEAR = 2024, 2025
FIRST_TRAIN_YEAR = 2019   # 2018 is only used as history for the circuit priors


def stint_table(laps):
    real = laps[laps['observed']]
    stints = real.groupby(['Year', 'circuit', 'race_id', 'Driver', 'Stint']).agg(
        compound=('Compound', 'first'), length=('LapNumber', 'size'),
        pitted=('pit', 'max'), retired=('retired_in_pits', 'max'), no_change=('no_tyre_change', 'max'))
    stints['pitted'] = stints['pitted'].astype(bool) & ~stints['retired'].astype(bool) & ~stints['no_change'].astype(bool)
    return stints.reset_index()


def add_circuit_priors(data, laps):
    """Typical stint length and number of stops per circuit, from earlier seasons only."""
    stints = stint_table(laps)
    finished = stints[stints['pitted'] & stints['compound'].isin(DRY)]
    stops = stints.groupby(['Year', 'circuit', 'race_id', 'Driver'])['pitted'].sum().reset_index()
    pit_losses = laps.loc[laps['pit_loss'].between(5, 60), ['Year', 'circuit', 'pit_loss']]

    parts = []
    for year, rows in data.groupby('Year'):
        past = finished[finished['Year'] < year]
        past_stops = stops[stops['Year'] < year]
        past_loss = pit_losses[pit_losses['Year'] < year]
        rows = rows.copy()
        if past.empty:
            rows[['expected_stint', 'circuit_stops', 'circuit_pit_loss']] = np.nan
            parts.append(rows)
            continue
        by_circuit_comp = past.groupby(['circuit', 'compound'])['length'].median()
        by_comp = past.groupby('compound')['length'].median()
        by_circuit = past.groupby('circuit')['length'].median()

        keys = pd.MultiIndex.from_arrays([rows['circuit'], rows['Compound']])
        expected = pd.Series(by_circuit_comp.reindex(keys).values, index=rows.index)
        expected = expected.fillna(rows['Compound'].map(by_comp))
        expected = expected.fillna(rows['circuit'].map(by_circuit)).fillna(past['length'].median())
        rows['expected_stint'] = expected
        rows['circuit_stops'] = rows['circuit'].map(past_stops.groupby('circuit')['pitted'].mean())
        rows['circuit_stops'] = rows['circuit_stops'].fillna(past_stops['pitted'].mean())
        rows['circuit_pit_loss'] = rows['circuit'].map(past_loss.groupby('circuit')['pit_loss'].median())
        rows['circuit_pit_loss'] = rows['circuit_pit_loss'].fillna(past_loss['pit_loss'].median())
        parts.append(rows)

    data = pd.concat(parts).sort_index()
    data['tyre_age_ratio'] = data['tyre_age'] / data['expected_stint']
    data['stint_laps_left'] = data['expected_stint'] - data['tyre_age']
    data['can_finish_on_tyres'] = (data['stint_laps_left'] >= data['laps_left']).astype(int)
    # positive margin: the car behind is far enough back to pit without losing the place
    data['stop_margin'] = data['gap_behind'] - data['circuit_pit_loss']
    return data


data = add_circuit_priors(data, laps)
data = data[data['Year'] >= FIRST_TRAIN_YEAR].reset_index(drop=True)

split = np.where(data['Year'] == TEST_YEAR, 'test', np.where(data['Year'] == VAL_YEAR, 'val', 'train'))
data['split'] = split
summary = data.groupby('split').agg(races=('race_id', 'nunique'), laps=('pit', 'size'), stops=('pit', 'sum'))
print(summary.loc[['train', 'val', 'test']])

## 5. Model
A multi-task network: shared residual MLP trunk (LayerNorm, SiLU, dropout) with a pit head and a compound head.
- Pit head: plain BCE, so outputs stay calibrated probabilities; the decision threshold is tuned later on 2024.
- Compound head: trained on **every lap** with the compound of the next stint as target (30× more examples than stop laps only); laps where the stop actually happens get weight 10.
- AdamW + one-cycle LR, gradient clipping, early stopping on validation PR-AUC, 5 seeds averaged.

In [ ]:
FEATURES = [
    # race and tyre state
    'lap', 'race_progress', 'laps_left', 'tyre_age', 'stops_done', 'fresh_tyre',
    'on_soft', 'on_medium', 'on_hard', 'on_intermediate', 'on_wet',
    'used_soft', 'used_medium', 'used_hard', 'dry_used', 'needs_second_dry',
    # pace and degradation
    'pace_delta', 'deg_rate', 'pace_vs_field',
    # race situation
    'position', 'gap_ahead', 'gap_behind', 'stop_margin', 'field_pit_share', 'field_pits_3',
    'field_ahead_on_stops', 'ahead_pitted', 'behind_pitted', 'teammate_pitted',
    'field_fitted_soft', 'field_fitted_medium', 'field_fitted_hard', 'field_fitted_intermediate',
    # track status
    'sc_last_lap', 'vsc_last_lap', 'sc_now', 'vsc_now', 'new_neutralisation',
    # weather
    'rain', 'track_temp', 'humidity', 'tyre_weather_mismatch', 'crossover_gain', 'odd_one_out',
    # circuit history
    'expected_stint', 'tyre_age_ratio', 'stint_laps_left', 'can_finish_on_tyres',
    'circuit_stops', 'circuit_pit_loss',
]
LIVE_STATUS = ['sc_now', 'vsc_now', 'new_neutralisation']
STOP_WEIGHT = 10.0


class Preprocessor:
    """Median imputation, clipping to the 0.5-99.5 % range and scaling, fitted on train only."""

    def fit(self, frame, cols):
        self.cols = cols
        values = frame[cols].astype(float)
        self.median = values.median()
        self.low, self.high = values.quantile(0.005), values.quantile(0.995)
        clipped = values.fillna(self.median).clip(self.low, self.high, axis=1)
        self.mean, self.std = clipped.mean(), clipped.std().replace(0, 1)
        return self

    def transform(self, frame):
        values = frame[self.cols].astype(float).fillna(self.median).clip(self.low, self.high, axis=1)
        return ((values - self.mean) / self.std).to_numpy(np.float32)


class ResidualBlock(nn.Module):
    def __init__(self, width, dropout):
        super().__init__()
        self.net = nn.Sequential(nn.LayerNorm(width), nn.Linear(width, width), nn.SiLU(),
                                 nn.Dropout(dropout), nn.Linear(width, width))

    def forward(self, x):
        return x + self.net(x)


class StrategyNet(nn.Module):
    def __init__(self, n_inputs, n_compounds, width=256, depth=3, dropout=0.15):
        super().__init__()
        self.stem = nn.Sequential(nn.Linear(n_inputs, width), nn.SiLU())
        self.blocks = nn.Sequential(*[ResidualBlock(width, dropout) for _ in range(depth)])
        self.norm = nn.LayerNorm(width)
        self.pit_head = nn.Sequential(nn.Linear(width, 64), nn.SiLU(), nn.Linear(64, 1))
        self.compound_head = nn.Sequential(nn.Linear(width, 64), nn.SiLU(), nn.Linear(64, n_compounds))

    def forward(self, x):
        h = self.norm(self.blocks(self.stem(x)))
        return self.pit_head(h).squeeze(-1), self.compound_head(h)


def make_tensors(frame, prep):
    x = torch.tensor(prep.transform(frame))
    pit = torch.tensor(frame['pit'].to_numpy(np.float32))
    compound = torch.tensor(frame['next_compound_id'].fillna(-1).to_numpy(np.int64))
    return x, pit, compound


@torch.no_grad()
def predict(model, x, batch_size=8192):
    model.eval()
    pit_out, comp_out = [], []
    for i in range(0, len(x), batch_size):
        p, c = model(x[i:i + batch_size].to(device))
        pit_out.append(torch.sigmoid(p).cpu())
        comp_out.append(torch.softmax(c, dim=1).cpu())
    return torch.cat(pit_out).numpy(), torch.cat(comp_out).numpy()


def train_model(train_df, val_df, cols, seed=42, epochs=30, batch_size=1024, lr=1e-3, weight_decay=1e-3,
                width=128, depth=2, dropout=0.3, compound_weight=0.5, patience=6, verbose=True):
    torch.manual_seed(seed)
    np.random.seed(seed)
    prep = Preprocessor().fit(train_df, cols)
    x_tr, pit_tr, comp_tr = make_tensors(train_df, prep)
    x_va, pit_va, comp_va = make_tensors(val_df, prep)

    model = StrategyNet(len(cols), len(COMPOUNDS), width, depth, dropout).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    steps = (len(x_tr) + batch_size - 1) // batch_size
    scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=lr, epochs=epochs, steps_per_epoch=steps)
    pit_loss_fn = nn.BCEWithLogitsLoss()
    comp_loss_fn = nn.CrossEntropyLoss(ignore_index=-1, label_smoothing=0.05, reduction='none')

    history, best_ap, best_state, bad_epochs = [], -1, None, 0
    for epoch in range(epochs):
        model.train()
        order = torch.randperm(len(x_tr))
        running = running_pit = 0.0
        for i in range(steps):
            idx = order[i * batch_size:(i + 1) * batch_size]
            xb, pb, cb = x_tr[idx].to(device), pit_tr[idx].to(device), comp_tr[idx].to(device)
            pit_logit, comp_logit = model(xb)
            pit_loss = pit_loss_fn(pit_logit, pb)
            loss = pit_loss
            # every lap knows which compound comes next; the laps where the stop happens count more
            known = cb >= 0
            if known.any():
                weights = (1 + (STOP_WEIGHT - 1) * pb)[known]
                comp_loss = comp_loss_fn(comp_logit[known], cb[known])
                loss = loss + compound_weight * (weights * comp_loss).sum() / weights.sum()
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            running += loss.item()
            running_pit += pit_loss.item()

        val_pit, val_comp = predict(model, x_va)
        val_loss = nn.functional.binary_cross_entropy(torch.tensor(val_pit).clamp(1e-7, 1 - 1e-7), pit_va).item()
        val_ap = average_precision_score(pit_va.numpy(), val_pit)
        at_stop = (pit_va.numpy() == 1) & (comp_va.numpy() >= 0)
        val_comp_acc = (val_comp[at_stop].argmax(1) == comp_va.numpy()[at_stop]).mean()
        history.append(dict(epoch=epoch + 1, train_loss=running / steps, train_pit_loss=running_pit / steps,
                            val_pit_loss=val_loss,
                            val_ap=val_ap, val_comp_acc=val_comp_acc))
        if verbose:
            print(f'epoch {epoch + 1:2d} | pit loss train {running_pit / steps:.4f} val {val_loss:.4f} '
                  f'| val PR-AUC {val_ap:.3f} | val compound acc {val_comp_acc:.1%}')
        if val_ap > best_ap:
            best_ap, bad_epochs = val_ap, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                if verbose:
                    print(f'early stop, best epoch {epoch + 1 - patience}')
                break

    model.load_state_dict(best_state)
    return model, prep, pd.DataFrame(history)

In [ ]:
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss


class EventScorer:
    """Turns per-lap probabilities into pit calls and scores them against the real stops.

    Alarm rule (causal, it never looks at future laps): call a stop on the first lap where
    p >= threshold, then stay quiet for `cooldown` laps. A new stint re-arms the alarm.
    Matching: each real stop can be claimed by one call within +-tolerance laps.
    """

    def __init__(self, frame):
        key = (frame['race_id'] + '|' + frame['Driver']).to_numpy()
        self.order = np.lexsort((frame['LapNumber'].to_numpy(), key))
        self.lap = frame['LapNumber'].to_numpy()[self.order]
        self.stint = frame['Stint'].to_numpy()[self.order]
        self.pit = frame['pit'].to_numpy()[self.order]
        key = key[self.order]
        self.bounds = np.r_[0, np.flatnonzero(key[1:] != key[:-1]) + 1, len(key)]

    def alarms(self, prob, threshold, cooldown):
        p = np.asarray(prob)[self.order]
        calls = np.zeros(len(p), dtype=bool)
        for a, b in zip(self.bounds[:-1], self.bounds[1:]):
            last_call, stint = -10 ** 6, None
            for i in range(a, b):
                if self.stint[i] != stint:
                    last_call, stint = -10 ** 6, self.stint[i]
                if p[i] >= threshold and self.lap[i] - last_call > cooldown:
                    calls[i] = True
                    last_call = self.lap[i]
        return calls

    def score(self, calls, tolerance=2):
        hits = 0
        for a, b in zip(self.bounds[:-1], self.bounds[1:]):
            stops = self.lap[a:b][self.pit[a:b] == 1]
            called = self.lap[a:b][calls[a:b]]
            free = np.ones(len(called), dtype=bool)
            for s in stops:
                dist = np.where(free, np.abs(called - s), np.inf)
                if len(dist) and dist.min() <= tolerance:
                    free[dist.argmin()] = False
                    hits += 1
        n_calls, n_stops = int(calls.sum()), int(self.pit.sum())
        precision = hits / n_calls if n_calls else 0.0
        recall = hits / n_stops if n_stops else 0.0
        f1 = 2 * precision * recall / (precision + recall) if hits else 0.0
        return dict(precision=precision, recall=recall, f1=f1, hits=hits, calls=n_calls, stops=n_stops)

    def tune(self, prob, tolerance=2):
        """Grid search over threshold and cooldown; only ever called on validation races."""
        best = None
        for threshold in np.unique(np.quantile(prob, np.linspace(0.90, 0.995, 40))):
            for cooldown in [1, 3, 5, 10]:
                result = self.score(self.alarms(prob, threshold, cooldown), tolerance)
                if best is None or result['f1'] > best['f1']:
                    best = dict(result, threshold=threshold, cooldown=cooldown)
        return best

    def to_frame_order(self, calls):
        out = np.zeros(len(calls), dtype=bool)
        out[self.order] = calls
        return out


def lap_metrics(y, score, is_probability=True):
    out = dict(pr_auc=average_precision_score(y, score), roc_auc=roc_auc_score(y, score))
    out['brier'] = brier_score_loss(y, score) if is_probability else np.nan
    return out

In [ ]:
train_df, val_df, test_df = (data[data['split'] == s].reset_index(drop=True) for s in ['train', 'val', 'test'])
N_SEEDS = 5

nets, histories = [], []
for i in range(N_SEEDS):
    print(f'--- seed {i + 1}/{N_SEEDS}')
    model, prep, history = train_model(train_df, val_df, FEATURES, seed=SEED + i, verbose=(i == 0))
    nets.append((model, prep))
    histories.append(history)
    print(f'best val PR-AUC {history.val_ap.max():.3f} at epoch {history.val_ap.idxmax() + 1}')


def net_predict(nets, frame):
    pit, comp = zip(*(predict(m, torch.tensor(p.transform(frame))) for m, p in nets))
    return np.mean(pit, axis=0), np.mean(comp, axis=0)


fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
for i, h in enumerate(histories):
    alpha = 1 if i == 0 else 0.25
    axes[0].plot(h.epoch, h.train_pit_loss, color=COLORS['rule'], alpha=alpha, label='train' if i == 0 else None)
    axes[0].plot(h.epoch, h.val_pit_loss, color=COLORS['net'], alpha=alpha, label='validation' if i == 0 else None)
    axes[1].plot(h.epoch, h.val_ap, color=COLORS['net'], alpha=alpha)
    axes[2].plot(h.epoch, h.val_comp_acc * 100, color=COLORS['gbm'], alpha=alpha)
    best = h.val_ap.idxmax()
    axes[1].scatter(h.epoch[best], h.val_ap[best], color=COLORS['net'], s=30, zorder=3, alpha=alpha)
axes[0].set(title='Pit loss (BCE)', xlabel='Epoch', ylim=(0.08, 0.16))
for ax in axes:
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
axes[0].legend()
axes[1].set(title='Validation PR-AUC (pit lap)', xlabel='Epoch')
axes[2].set(title='Validation compound accuracy at stops', xlabel='Epoch')
axes[2].yaxis.set_major_formatter(mticker.PercentFormatter())
fig.suptitle(f'Training, {N_SEEDS} seeds (dots = checkpoint kept by early stopping)', x=0.01, ha='left',
             color=MUTED, fontsize=10)
fig.tight_layout()
save(fig, 'training_curves')
plt.show()

## 6. Baselines

In [ ]:
import lightgbm as lgb

GBM_PARAMS = dict(n_estimators=3000, learning_rate=0.03, num_leaves=31, min_child_samples=50, subsample=0.8,
                  subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0, random_state=SEED, verbose=-1)
at_stop = lambda f: f[(f['pit'] == 1) & f['next_compound_id'].notna()]

# 1) gradient boosting: strong tabular baseline, same features
gbm_pit = lgb.LGBMClassifier(**GBM_PARAMS)
gbm_pit.fit(train_df[FEATURES], train_df['pit'], eval_set=[(val_df[FEATURES], val_df['pit'])],
            eval_metric='average_precision', callbacks=[lgb.early_stopping(100, verbose=False)])
labelled = train_df[train_df['next_compound_id'].notna()]
gbm_comp = lgb.LGBMClassifier(**GBM_PARAMS)
gbm_comp.fit(labelled[FEATURES], labelled['next_compound_id'].astype(int),
             sample_weight=np.where(labelled['pit'] == 1, STOP_WEIGHT, 1),
             eval_set=[(at_stop(val_df)[FEATURES], at_stop(val_df)['next_compound_id'].astype(int))],
             callbacks=[lgb.early_stopping(100, verbose=False)])


def gbm_predict(frame):
    comp = np.zeros((len(frame), len(COMPOUNDS)))
    comp[:, gbm_comp.classes_] = gbm_comp.predict_proba(frame[FEATURES])
    return gbm_pit.predict_proba(frame[FEATURES])[:, 1], comp


# 2) the same network on the inputs the first version of this project had
BASIC_FEATURES = ['lap', 'race_progress', 'laps_left', 'tyre_age', 'on_soft', 'on_medium', 'on_hard',
                  'on_intermediate', 'on_wet', 'pace_delta', 'position', 'sc_now', 'vsc_now', 'rain', 'track_temp']
basic_nets = [train_model(train_df, val_df, BASIC_FEATURES, seed=SEED + i, verbose=False)[:2] for i in range(N_SEEDS)]

# 3) pit when the tyres reach the usual stint length for this circuit,
#    fit the compound most teams switched to from the current one
usual_next = at_stop(train_df).groupby('Compound')['next_compound'].agg(lambda s: s.mode()[0])


def rule_predict(frame):
    comp = np.zeros((len(frame), len(COMPOUNDS)))
    choice = frame['Compound'].map(usual_next).fillna('HARD').map(COMPOUNDS.index)
    comp[np.arange(len(frame)), choice.to_numpy()] = 1
    return frame['tyre_age_ratio'].fillna(0).to_numpy(), comp


predictions = {}
for split_name, frame in [('val', val_df), ('test', test_df)]:
    net_pit, net_comp = net_predict(nets, frame)
    gbm_pit_p, gbm_comp_p = gbm_predict(frame)
    predictions[split_name] = {
        'Tyre-age rule': rule_predict(frame),
        'Net, basic inputs': net_predict(basic_nets, frame),
        'LightGBM': (gbm_pit_p, gbm_comp_p),
        'Network (5 seeds)': (net_pit, net_comp),
        'Network + LightGBM': ((net_pit + gbm_pit_p) / 2, (net_comp + gbm_comp_p) / 2),
    }
MODEL_COLORS = {'Tyre-age rule': COLORS['rule'], 'Net, basic inputs': COLORS['original'],
                'LightGBM': COLORS['gbm'], 'Network (5 seeds)': '#6250d6', 'Network + LightGBM': COLORS['net']}
print('trees used: pit', gbm_pit.best_iteration_, '| compound', gbm_comp.best_iteration_)

## 7. Results on the held-out 2025 season
- **PR-AUC / ROC-AUC / Brier**: per-lap quality of the pit probability, independent of any threshold.
- **Precision / recall / F1**: pit *calls* from the alarm rule, matched to real stops within ±2 laps. Threshold and cooldown are chosen on 2024.
- **compound_acc**: accuracy of the predicted compound at the real stops.

In [ ]:
from sklearn.metrics import precision_recall_curve

val_events, test_events = EventScorer(val_df), EventScorer(test_df)
TOLERANCES = [0, 1, 2, 3]
rows, tolerance_f1, settings = [], {}, {}
for name in predictions['test']:
    val_pit, test_pit = predictions['val'][name][0], predictions['test'][name][0]
    is_prob = name != 'Tyre-age rule'
    f1_by_tol = []
    for tol in TOLERANCES:
        tuned = val_events.tune(val_pit, tol)          # threshold + cooldown picked on 2024 only
        result = test_events.score(test_events.alarms(test_pit, tuned['threshold'], tuned['cooldown']), tol)
        f1_by_tol.append(result['f1'])
        if tol == 2:
            settings[name] = tuned
            main = result
    tolerance_f1[name] = f1_by_tol
    comp_true = at_stop(test_df)['next_compound_id'].to_numpy()
    comp_pred = predictions['test'][name][1][(test_df['pit'] == 1) & test_df['next_compound_id'].notna()].argmax(1)
    rows.append(dict(model=name, **lap_metrics(test_df['pit'], test_pit, is_prob),
                     precision=main['precision'], recall=main['recall'], f1=main['f1'],
                     calls=main['calls'], compound_acc=(comp_pred == comp_true).mean()))

results = pd.DataFrame(rows).set_index('model')
print(f"Test season {TEST_YEAR}: {test_df.race_id.nunique()} races, {int(test_df.pit.sum())} stops "
      f"(pit-lap base rate {test_df.pit.mean():.1%}). Event metrics: +-2 laps.")
display(results.style.format({'pr_auc': '{:.3f}', 'roc_auc': '{:.3f}', 'brier': '{:.4f}', 'precision': '{:.1%}',
                              'recall': '{:.1%}', 'f1': '{:.3f}', 'calls': '{:.0f}', 'compound_acc': '{:.1%}'},
                             na_rep='-'))
results.to_csv(os.path.join(OUT_DIR, 'test_results.csv'))

# precision-recall per lap, calibration, and event F1 against the matching tolerance
fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
for name, (pit_prob, _) in predictions['test'].items():
    prec, rec, _ = precision_recall_curve(test_df['pit'], pit_prob)
    axes[0].plot(rec, prec, color=MODEL_COLORS[name], lw=1.6, label=f"{name} ({results.loc[name, 'pr_auc']:.3f})")
axes[0].axhline(test_df['pit'].mean(), color=MUTED, ls=':', lw=1)
axes[0].text(0.01, test_df['pit'].mean() + 0.01, 'random guess', color=MUTED, fontsize=8)
axes[0].set(title='Pit lap: precision-recall', xlabel='Recall', ylabel='Precision', xlim=(0, 1), ylim=(0, 1))
axes[0].legend(fontsize=8, title='PR-AUC', title_fontsize=8)

bins = np.array([0, 0.01, 0.02, 0.05, 0.1, 0.2, 0.3, 0.5, 1.0])
for name in ['LightGBM', 'Network (5 seeds)', 'Network + LightGBM']:
    pit_prob = predictions['test'][name][0]
    bucket = np.digitize(pit_prob, bins[1:-1])
    table = pd.DataFrame({'p': pit_prob, 'y': test_df['pit'], 'b': bucket}).groupby('b').agg(p=('p', 'mean'), y=('y', 'mean'), n=('y', 'size'))
    table = table[table.n >= 30]
    axes[1].plot(table.p, table.y, marker='o', ms=5, color=MODEL_COLORS[name], lw=1.6, label=name)
axes[1].plot([0, 0.6], [0, 0.6], color=MUTED, ls='--', lw=1)
axes[1].set(title='Calibration', xlabel='Predicted pit probability', ylabel='Observed pit rate',
            xlim=(0, 0.6), ylim=(0, 0.6))
axes[1].legend(fontsize=8)

width = 0.8 / len(tolerance_f1)
for i, (name, f1s) in enumerate(tolerance_f1.items()):
    axes[2].bar(np.arange(len(TOLERANCES)) + (i - (len(tolerance_f1) - 1) / 2) * width, f1s, width * 0.92,
                color=MODEL_COLORS[name], label=name)
axes[2].set_xticks(range(len(TOLERANCES)), [f'exact' if t == 0 else f'±{t}' for t in TOLERANCES])
axes[2].set(title='Pit-stop calls: event F1', xlabel='Allowed timing error (laps)', ylabel='F1', ylim=(0, 0.7))
axes[2].grid(axis='x', visible=False)
fig.suptitle(f'Held-out season {TEST_YEAR}', x=0.01, ha='left', color=MUTED, fontsize=10)
fig.tight_layout()
save(fig, 'model_comparison')
plt.show()

## 8. Compound choice

In [ ]:
BEST = 'Network + LightGBM'
stops = at_stop(test_df)
mask = ((test_df['pit'] == 1) & test_df['next_compound_id'].notna()).to_numpy()
true_id = stops['next_compound_id'].astype(int).to_numpy()
pred_id = predictions['test'][BEST][1][mask].argmax(1)
present = [i for i in range(len(COMPOUNDS)) if (true_id == i).any() or (pred_id == i).any()]
labels = [COMPOUNDS[i].title() for i in present]

counts = pd.crosstab(pd.Categorical(true_id, present), pd.Categorical(pred_id, present), dropna=False).to_numpy()
share = counts / counts.sum(axis=1, keepdims=True).clip(min=1)

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(12.5, 4.4), gridspec_kw=dict(width_ratios=[1.25, 1]))
ax0.imshow(share, cmap='Blues', vmin=0, vmax=1)
for r in range(len(present)):
    for c in range(len(present)):
        ax0.text(c, r, f'{share[r, c]:.0%}\n({counts[r, c]})', ha='center', va='center', fontsize=8,
                 color='white' if share[r, c] > 0.55 else INK)
ax0.set_xticks(range(len(present)), labels)
ax0.set_yticks(range(len(present)), labels)
ax0.set(xlabel='Predicted compound', ylabel='Compound actually fitted',
        title=f'Next compound at real stops ({BEST})')
ax0.grid(False)

names = list(predictions['test'])
accs = [results.loc[n, 'compound_acc'] * 100 for n in names]
bars = ax1.barh(names, accs, color=[MODEL_COLORS[n] for n in names], height=0.6)
ax1.bar_label(bars, fmt='%.1f%%', padding=4, color=INK, fontsize=9)
ax1.axvline((true_id == COMPOUNDS.index('HARD')).mean() * 100, color=MUTED, ls=':', lw=1)
ax1.text((true_id == COMPOUNDS.index('HARD')).mean() * 100 + 1, -0.6, 'always HARD', color=MUTED, fontsize=8)
ax1.set(title='Accuracy at real stops', xlim=(0, 100))
ax1.xaxis.set_major_formatter(mticker.PercentFormatter())
ax1.invert_yaxis()
ax1.grid(axis='y', visible=False)
fig.tight_layout()
save(fig, 'compound_prediction')
plt.show()

## 9. Race replay
A typical test race is chosen automatically (per-race F1 closest to the median)

In [ ]:
best_settings = settings[BEST]
test_df['p_pit'] = predictions['test'][BEST][0]
test_df['call'] = test_events.to_frame_order(
    test_events.alarms(test_df['p_pit'].to_numpy(), best_settings['threshold'], best_settings['cooldown']))

# show a typical race, not the best one: the race whose F1 is closest to the median
per_race = {}
for race, frame in test_df.groupby('race_id'):
    scorer = EventScorer(frame)
    per_race[race] = scorer.score(scorer.alarms(frame['p_pit'].to_numpy(), best_settings['threshold'],
                                                best_settings['cooldown']))['f1']
per_race = pd.Series(per_race)
race = (per_race - per_race.median()).abs().idxmin()
frame = test_df[test_df['race_id'] == race]
print(f'Per-race F1: median {per_race.median():.2f}, range {per_race.min():.2f}-{per_race.max():.2f}. '
      f'Showing {race} (F1 {per_race[race]:.2f}).')

# whole field at a glance
order = frame.groupby('Driver')['position'].median().sort_values().index
grid = frame.pivot_table(index='Driver', columns='LapNumber', values='p_pit').reindex(order)
fig, ax = plt.subplots(figsize=(13, 0.28 * len(order) + 1.4))
image = ax.imshow(grid.to_numpy(), aspect='auto', cmap='Blues', vmin=0, vmax=0.5,
                  extent=[grid.columns.min() - 0.5, grid.columns.max() + 0.5, len(order) - 0.5, -0.5])
row = {d: i for i, d in enumerate(order)}
real = frame[frame['pit'] == 1]
called = frame[frame['call']]
ax.scatter(real['LapNumber'], real['Driver'].map(row), marker='o', s=34, facecolor='none',
           edgecolor=INK, lw=1.4, label='real stop')
ax.scatter(called['LapNumber'], called['Driver'].map(row), marker='x', s=22, color=COLORS['gbm'],
           lw=1.6, label='model calls a stop')
neutral = frame.groupby('LapNumber')[['sc_now', 'vsc_now']].max().sum(axis=1) > 0
for lap in neutral[neutral].index:
    ax.axvspan(lap - 0.5, lap + 0.5, color='#eda100', alpha=0.12, lw=0)
ax.set_yticks(range(len(order)), order, fontsize=8)
ax.set(xlabel='Lap', title=f'{race}: pit probability for every driver (yellow = SC / VSC)')
ax.grid(False)
fig.colorbar(image, ax=ax, pad=0.01, fraction=0.02, label='P(pit this lap)')
ax.legend(loc='upper left', bbox_to_anchor=(0, -0.12), ncol=2, fontsize=8)
save(fig, 'race_heatmap')
plt.show()

# three drivers in detail (the busiest ones: most real stops + calls), tyre stints in the background
busy = frame.groupby('Driver')['pit'].sum() + frame.groupby('Driver')['call'].sum()
drivers = busy.sort_values(ascending=False).index[:3]
fig, axes = plt.subplots(len(drivers), 1, figsize=(12, 2.4 * len(drivers)), sharex=True)
for ax, driver in zip(axes, drivers):
    d = frame[frame['Driver'] == driver].sort_values('LapNumber')
    for _, stint in d.groupby('Stint'):
        ax.axvspan(stint['LapNumber'].min() - 0.5, stint['LapNumber'].max() + 0.5,
                   color=TYRE_COLORS.get(stint['Compound'].iloc[0], GRID), alpha=0.15, lw=0)
        if len(stint) >= 4:
            ax.text(stint['LapNumber'].min(), 0.56, str(stint['Compound'].iloc[0]).title(), fontsize=8, color=MUTED)
    ax.plot(d['LapNumber'], d['p_pit'], color=COLORS['net'], lw=1.8)
    ax.axhline(best_settings['threshold'], color=MUTED, ls='--', lw=1)
    for lap in d.loc[d['pit'] == 1, 'LapNumber']:
        ax.axvline(lap, color=INK, lw=1.2)
    ax.scatter(d.loc[d['call'], 'LapNumber'], d.loc[d['call'], 'p_pit'], marker='v', s=45,
               color=COLORS['gbm'], zorder=3)
    ax.set(ylim=(0, max(0.62, d['p_pit'].max() * 1.1)), ylabel='P(pit)')
    ax.set_title(driver, loc='left', fontsize=10)
axes[-1].set_xlabel('Lap')
handles = [plt.Line2D([], [], color=COLORS['net'], lw=1.8, label='pit probability'),
           plt.Line2D([], [], color=INK, lw=1.2, label='real stop'),
           plt.Line2D([], [], color=COLORS['gbm'], marker='v', ls='', label='model calls a stop'),
           plt.Line2D([], [], color=MUTED, ls='--', lw=1, label='threshold (tuned on 2024)')]
axes[0].legend(handles=handles, ncol=4, loc='lower left', bbox_to_anchor=(0, 1.12), fontsize=8)
fig.suptitle(f'{race}: race replay', x=0.01, ha='left', y=1.02, fontsize=12, fontweight='semibold')
fig.tight_layout()
save(fig, 'race_replay')
plt.show()

## 10. What drives the predictions

In [ ]:
FEATURE_GROUPS = {
    'Tyres & stint': ['tyre_age', 'stops_done', 'fresh_tyre', 'on_soft', 'on_medium', 'on_hard', 'on_intermediate',
                      'on_wet', 'used_soft', 'used_medium', 'used_hard', 'dry_used', 'needs_second_dry'],
    'Race phase': ['lap', 'race_progress', 'laps_left'],
    'Pace & degradation': ['pace_delta', 'deg_rate', 'pace_vs_field'],
    'Rivals & field': ['position', 'gap_ahead', 'gap_behind', 'stop_margin', 'field_pit_share', 'field_pits_3',
                       'field_ahead_on_stops', 'ahead_pitted', 'behind_pitted', 'teammate_pitted',
                       'field_fitted_soft', 'field_fitted_medium', 'field_fitted_hard', 'field_fitted_intermediate'],
    'Safety car': ['sc_last_lap', 'vsc_last_lap', 'sc_now', 'vsc_now', 'new_neutralisation'],
    'Weather': ['rain', 'track_temp', 'humidity', 'tyre_weather_mismatch', 'crossover_gain', 'odd_one_out'],
    'Circuit history': ['expected_stint', 'tyre_age_ratio', 'stint_laps_left', 'can_finish_on_tyres',
                        'circuit_stops', 'circuit_pit_loss'],
}
assert sorted(sum(FEATURE_GROUPS.values(), [])) == sorted(FEATURES)

# permutation importance on the validation races: shuffle one group of columns together
# and measure how much PR-AUC and compound accuracy drop
rng = np.random.default_rng(SEED)
val_mask = ((val_df['pit'] == 1) & val_df['next_compound_id'].notna()).to_numpy()
val_true = val_df.loc[val_mask, 'next_compound_id'].astype(int).to_numpy()
base_pit, base_comp = net_predict(nets, val_df)
base_ap = average_precision_score(val_df['pit'], base_pit)
base_acc = (base_comp[val_mask].argmax(1) == val_true).mean()

drops = []
for group, cols in FEATURE_GROUPS.items():
    shuffled = val_df.copy()
    perm = rng.permutation(len(val_df))
    shuffled[cols] = val_df[cols].to_numpy()[perm]
    pit, comp = net_predict(nets, shuffled)
    drops.append(dict(group=group, pit=base_ap - average_precision_score(val_df['pit'], pit),
                      compound=(base_acc - (comp[val_mask].argmax(1) == val_true).mean()) * 100))
drops = pd.DataFrame(drops).set_index('group')

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), sharey=True)
for ax, col, title, color, fmt in [(axes[0], 'pit', 'Pit timing: drop in PR-AUC', COLORS['net'], '%.3f'),
                                   (axes[1], 'compound', 'Compound choice: drop in accuracy (pp)', COLORS['gbm'], '%.1f')]:
    values = drops[col].sort_values()
    bars = ax.barh(values.index, values.values, color=color, height=0.6)
    ax.bar_label(bars, fmt=fmt, padding=3, fontsize=8, color=INK)
    ax.set_title(title)
    ax.grid(axis='y', visible=False)
    ax.axvline(0, color=MUTED, lw=0.8)
fig.suptitle('What the network relies on (shuffle one feature group at a time, validation 2024)',
             x=0.01, ha='left', color=MUTED, fontsize=10)
fig.tight_layout()
save(fig, 'feature_importance')
plt.show()

## 11. Save model and environment

In [ ]:
import joblib
from importlib.metadata import version

model_dir = os.path.join(BASE_DIR, 'model')
os.makedirs(model_dir, exist_ok=True)
for i, (model, prep) in enumerate(nets):
    torch.save(model.state_dict(), os.path.join(model_dir, f'strategy_net_seed{i}.pth'))
    joblib.dump(prep, os.path.join(model_dir, f'preprocessor_seed{i}.pkl'))
joblib.dump({'pit': gbm_pit, 'compound': gbm_comp, 'features': FEATURES, 'compounds': COMPOUNDS,
             'alarm': settings[BEST]}, os.path.join(model_dir, 'lightgbm_and_config.pkl'))
print('saved to', model_dir)
print(' '.join(f'{p}=={version(p)}' for p in
               ['torch', 'pandas', 'numpy', 'scikit-learn', 'lightgbm', 'matplotlib', 'fastf1', 'tqdm']))